In [7]:
openaq_api_key = ""
date_from = "2024-01-01T00:00:00Z"
date_to = "2024-04-01T00:00:00Z"
bbox = "-74.26,40.49,-73.70,40.92"   # New York: lon_min, lat_min, lon_max, lat_max
parameters = ["pm25", "no2"]

StatementMeta(, df4787e4-127c-4e77-ba80-26fe74d706eb, 8, Finished, Available, Finished, False)

In [8]:
import requests, time, json, os
from datetime import datetime, timezone

BASE = "https://api.openaq.org/v3"
session = requests.Session()
session.headers.update({"X-API-Key": openaq_api_key})
def get_all(path, params):
    results, page = [], 1
    while True:
        p = {**params, "limit": 1000, "page": page}
        for attempt in range(5):
            r = session.get(f"{BASE}{path}", params=p, timeout=60)
            if r.status_code == 429:
                time.sleep(int(r.headers.get("x-ratelimit-reset", 30)))
                continue
            r.raise_for_status()
            break
        batch = r.json().get("results", [])
        results.extend(batch)
        time.sleep(1.1)
        if len(batch) < 1000:
            return results
        page += 1

StatementMeta(, df4787e4-127c-4e77-ba80-26fe74d706eb, 9, Finished, Available, Finished, False)

In [9]:
locations = get_all("/locations", {"bbox": bbox})
print(f"Locations in bbox: {len(locations)}")
sensors = []
for loc in locations:
    first = (loc.get("datetimeFirst") or {}).get("utc")
    last = (loc.get("datetimeLast") or {}).get("utc")
    if not first or not last or last < date_from or first > date_to:
        continue
    for s in loc.get("sensors", []):
        if s["parameter"]["name"] in parameters:
            sensors.append({
                "location_id": loc["id"],
                "location_name": loc.get("name"),
                "latitude": loc["coordinates"]["latitude"],
                "longitude": loc["coordinates"]["longitude"],
                "sensor_id": s["id"],
                "parameter": s["parameter"]["name"],
                "units": s["parameter"]["units"],
            })
print(f"Active PM2.5/NO2 sensors in period: {len(sensors)}")

StatementMeta(, df4787e4-127c-4e77-ba80-26fe74d706eb, 10, Finished, Available, Finished, False)

Locations in bbox: 76
Active PM2.5/NO2 sensors in period: 28


In [10]:
rows, raw = [], {}
for i, s in enumerate(sensors, 1):
    days = get_all(f"/sensors/{s['sensor_id']}/days", {
        "date_from": date_from[:10], "date_to": date_to[:10],
        "datetime_from": date_from, "datetime_to": date_to,
    })
    days = [d for d in days
            if date_from <= d["period"]["datetimeFrom"]["utc"] < date_to]
    raw[s["sensor_id"]] = days
    for d in days:
        summary = d.get("summary") or {}
        rows.append({
            **s,
            "date_utc": d["period"]["datetimeFrom"]["utc"],
            "value_avg": d.get("value"),
            "value_min": summary.get("min"),
            "value_max": summary.get("max"),
            "coverage_pct": (d.get("coverage") or {}).get("percentComplete"),
        })
    print(f"{i}/{len(sensors)} sensor {s['sensor_id']} ({s['parameter']}): {len(days)} days")
print(f"Total rows: {len(rows)}")

StatementMeta(, df4787e4-127c-4e77-ba80-26fe74d706eb, 11, Finished, Available, Finished, False)

1/28 sensor 673 (pm25): 90 days
2/28 sensor 1097 (pm25): 91 days
3/28 sensor 1102 (pm25): 90 days
4/28 sensor 1103 (pm25): 91 days
5/28 sensor 1152 (pm25): 91 days
6/28 sensor 1128 (pm25): 91 days
7/28 sensor 1145 (pm25): 91 days
8/28 sensor 1146 (pm25): 91 days
9/28 sensor 1535 (no2): 91 days
10/28 sensor 1534 (pm25): 84 days
11/28 sensor 5077566 (pm25): 91 days
12/28 sensor 3951 (no2): 87 days
13/28 sensor 1758 (pm25): 74 days
14/28 sensor 23341 (no2): 91 days
15/28 sensor 2644 (no2): 91 days
16/28 sensor 25520 (pm25): 0 days
17/28 sensor 1662910 (pm25): 91 days
18/28 sensor 2165515 (pm25): 89 days
19/28 sensor 2189530 (pm25): 91 days
20/28 sensor 2780575 (pm25): 91 days
21/28 sensor 7324143 (pm25): 90 days
22/28 sensor 7386938 (pm25): 90 days
23/28 sensor 7763001 (pm25): 27 days
24/28 sensor 7979399 (pm25): 0 days
25/28 sensor 7815388 (pm25): 55 days
26/28 sensor 7897339 (pm25): 40 days
27/28 sensor 7897355 (pm25): 40 days
28/28 sensor 8520925 (pm25): 14 days
Total rows: 2053


In [11]:
import pandas as pd

run_ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
os.makedirs("/lakehouse/default/Files/raw/openaq", exist_ok=True)
with open(f"/lakehouse/default/Files/raw/openaq/openaq_{run_ts}.json", "w") as f:
    json.dump({"locations": locations, "measurements": raw}, f)
df = spark.createDataFrame(pd.DataFrame(rows))
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze_openaq")
spark.createDataFrame(pd.DataFrame(sensors)) \
     .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze_openaq_sensors")
display(df.limit(10))

StatementMeta(, df4787e4-127c-4e77-ba80-26fe74d706eb, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 62a9a5bc-8fc3-426b-9a2f-debefd3e250c)